# AgriAssist Crop Disease Detection Model Training

This notebook trains a MobileNetV2 model for crop disease detection using the PlantVillage dataset from Kaggle.

**Instructions:**
1. Upload your `kaggle.json` file (from Kaggle account settings) to Colab's file explorer
2. Enable GPU runtime: Runtime > Change runtime type > GPU
3. Run all cells sequentially
4. Download the generated zip file at the end
5. Extract and place the files in your local project's `/ml/models/` directory

In [ ]:
# Install required packages
!pip install -q tensorflow kaggle pillow numpy

In [ ]:
# Setup Kaggle API
import os
import shutil
from google.colab import files

# Create .kaggle directory
!mkdir -p ~/.kaggle

# Upload kaggle.json if not already present
if not os.path.exists('kaggle.json'):
    print("Please upload your kaggle.json file")
    uploaded = files.upload()
    for fn in uploaded.keys():
        shutil.move(fn, 'kaggle.json')

# Copy to .kaggle directory
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API configured successfully")

In [ ]:
# Download PlantVillage dataset
!kaggle datasets download -d emmarex/plantdisease -p /content/dataset --unzip

print("Dataset downloaded successfully")

In [ ]:
# Define selected classes (8-10 classes for lightweight model)
SELECTED_CLASSES = [
    "Tomato___healthy",
    "Tomato___Late_blight",
    "Tomato___Early_blight",
    "Potato___healthy",
    "Potato___Late_blight",
    "Potato___Early_blight",
    "Corn_(maize)___healthy",
    "Corn_(maize)___Common_rust",
    "Pepper,_bell___healthy",
    "Pepper,_bell___Bacterial_spot"
]

print(f"Selected {len(SELECTED_CLASSES)} classes for training")

In [ ]:
# Sample and organize dataset
import os
import shutil
from pathlib import Path
import random

dataset_path = Path("/content/dataset/PlantVillage")
output_path = Path("/content/processed_dataset")

MAX_IMAGES_PER_CLASS = 300

# Create output directories
output_path.mkdir(exist_ok=True)
for class_name in SELECTED_CLASSES:
    (output_path / class_name).mkdir(exist_ok=True)

# Copy images for selected classes
for class_name in SELECTED_CLASSES:
    class_path = dataset_path / class_name
    if class_path.exists():
        images = list(class_path.glob("*.JPG")) + list(class_path.glob("*.jpg")) + list(class_path.glob("*.png"))
        
        # Sample max 300 images per class
        if len(images) > MAX_IMAGES_PER_CLASS:
            images = random.sample(images, MAX_IMAGES_PER_CLASS)
        
        # Copy images
        for img_path in images:
            shutil.copy(img_path, output_path / class_name / img_path.name)
        
        print(f"{class_name}: {len(images)} images")
    else:
        print(f"Warning: {class_name} not found in dataset")

print("\nDataset organized successfully")

In [ ]:
# Split into train/validation
from sklearn.model_selection import train_test_split

train_path = Path("/content/train_dataset")
val_path = Path("/content/val_dataset")

train_path.mkdir(exist_ok=True)
val_path.mkdir(exist_ok=True)

for class_name in SELECTED_CLASSES:
    class_images = list((output_path / class_name).glob("*"))
    train_imgs, val_imgs = train_test_split(class_images, test_size=0.2, random_state=42)
    
    (train_path / class_name).mkdir(exist_ok=True)
    (val_path / class_name).mkdir(exist_ok=True)
    
    for img in train_imgs:
        shutil.copy(img, train_path / class_name / img.name)
    for img in val_imgs:
        shutil.copy(img, val_path / class_name / img.name)
    
    print(f"{class_name}: {len(train_imgs)} train, {len(val_imgs)} val")

print("\nTrain/validation split completed")

In [ ]:
# Load and preprocess data
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

IMG_SIZE = (160, 160)
BATCH_SIZE = 32

train_ds = image_dataset_from_directory(
    train_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = image_dataset_from_directory(
    val_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Configure performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

print(f"Training batches: {len(train_ds)}")
print(f"Validation batches: {len(val_ds)}")

In [ ]:
# Build MobileNetV2 model with transfer learning
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, callbacks

# Load pre-trained MobileNetV2
base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze the base model
base_model.trainable = False

# Add custom head
inputs = tf.keras.Input(shape=(160, 160, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(SELECTED_CLASSES), activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Train the model
EPOCHS = 8

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

print("Training completed")

In [ ]:
# Evaluate the model
val_loss, val_accuracy = model.evaluate(val_ds)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
# Save model and metadata
import json
from datetime import datetime

# Create models directory
!mkdir -p /content/models

# Save model
model.save('/content/models/disease_model.h5')
print("Model saved as disease_model.h5")

# Save class labels
with open('/content/models/class_labels.json', 'w') as f:
    json.dump(SELECTED_CLASSES, f, indent=2)
print("Class labels saved as class_labels.json")

# Save metadata
metadata = {
    "model_name": "AgriAssist Disease Detection Model",
    "model_type": "MobileNetV2 Transfer Learning",
    "training_date": datetime.now().isoformat(),
    "num_classes": len(SELECTED_CLASSES),
    "classes": SELECTED_CLASSES,
    "image_size": IMG_SIZE,
    "max_images_per_class": MAX_IMAGES_PER_CLASS,
    "epochs_trained": len(history.history['loss']),
    "final_val_accuracy": float(val_accuracy),
    "final_val_loss": float(val_loss),
    "dataset_source": "Kaggle PlantVillage (emmarex/plantdisease)"
}

with open('/content/models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("Metadata saved as metadata.json")

In [ ]:
# Create zip file for download
import zipfile

zip_path = "/content/agriassist_disease_model.zip"

with zipfile.ZipFile(zip_path, 'w') as zipf:
    for file in ['disease_model.h5', 'class_labels.json', 'metadata.json']:
        zipf.write(f'/content/models/{file}', file)

print(f"Zip file created: {zip_path}")

In [ ]:
# Download the zip file
from google.colab import files
files.download(zip_path)

print("\n=== NEXT STEPS ===")
print("1. Download the zip file")
print("2. Extract the zip file")
print("3. Copy the 3 files (disease_model.h5, class_labels.json, metadata.json) to your local project's /ml/models/ directory")
print("4. The backend will automatically use the trained model on next restart")